# Computrabajo Lima: scraping con Selenium + BeautifulSoup
Flujo: **Selenium** carga las páginas (y espera el JS) → `page_source` se pasa a **BeautifulSoup** → se segmenta el texto por encabezados.

In [1]:
import os
import random
import re
from time import sleep
from urllib.parse import urljoin

import pandas as pd
from bs4 import BeautifulSoup as bs
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    NoSuchElementException, ElementNotInteractableException, TimeoutException
)

In [2]:
BASE_URL = 'https://pe.computrabajo.com'
URL_BUSQUEDA = f'{BASE_URL}/empleos-de-ingenieria-y-tecnico-en-lima'  # sin ?p=
MAX_PAGINAS = 200  # tope de seguridad
LIMITE_EXTRACCION = None  # número de ofertas a extraer; None = todas las de la página

PATRON_HEADING_DESCRIPCION = re.compile(r'Descripci[oó]n de la oferta', re.IGNORECASE)
PATRON_HEADING_REQUISITOS = re.compile(r'Requerimientos|Requisitos', re.IGNORECASE)
PATRON_HEADING_CORTE_FINAL = re.compile(r'Postularme|Denunciar empleo|Compartir', re.IGNORECASE)

PATRON_SALARIO = re.compile(
    r'S/\.?\s?[\d][\d.,]*(?:\s*(?:a|-|hasta)\s*S/\.?\s?[\d][\d.,]*)?', re.IGNORECASE
)
PATRON_MODALIDAD = re.compile(r'\b(Remoto|Presencial|H[ií]brido|Semipresencial)\b', re.IGNORECASE)

In [3]:
def dividir_bloques(texto_completo: str, titulo: str) -> dict:
    """Divide el texto de la oferta en resumen / descripción / requisitos."""
    resumen, descripcion, requisitos = "", "", ""

    idx_titulo = texto_completo.find(titulo) if titulo else -1
    match_desc = PATRON_HEADING_DESCRIPCION.search(texto_completo)
    match_req = PATRON_HEADING_REQUISITOS.search(
        texto_completo, match_desc.end() if match_desc else 0
    )

    if idx_titulo != -1 and match_desc:
        resumen = texto_completo[idx_titulo + len(titulo):match_desc.start()].strip()
    elif match_desc:
        resumen = texto_completo[:match_desc.start()].strip()

    if match_desc:
        fin_desc = match_req.start() if match_req else len(texto_completo)
        descripcion = texto_completo[match_desc.end():fin_desc].strip()

    if match_req:
        resto = texto_completo[match_req.end():]
        corte = PATRON_HEADING_CORTE_FINAL.search(resto)
        requisitos = resto[:corte.start()].strip() if corte else resto.strip()

    return {"resumen": resumen, "descripcion": descripcion, "requisitos": requisitos}


def buscar_con_prioridad(patron, *bloques):
    """Busca el patrón primero en Resumen y, si no aparece, en Descripción."""
    for bloque in bloques:
        if bloque:
            m = patron.search(bloque)
            if m:
                return m.group(0).strip()
    return None


def texto_visible(html: str) -> str:
    """Equivalente aproximado a inner_text() de Playwright usando bs4."""
    soup = bs(html, 'html.parser')
    for t in soup(['script', 'style', 'noscript']):
        t.decompose()
    return soup.body.get_text('\n', strip=True) if soup.body else ''

In [4]:
options = webdriver.ChromeOptions()
options.add_argument('--start-maximized')

driver = webdriver.Chrome(options=options)
driver.set_page_load_timeout(60)

In [5]:
PAGINA_INICIO = 1   # para retomar tras un fallo: pon el nº de página (sin reiniciar el kernel)

if PAGINA_INICIO == 1 or 'urls_empleos' not in globals():
    urls_empleos, vistos = [], set()

def cerrar_banner():
    try:
        driver.find_element(By.XPATH, "//*[self::button or self::a][contains(., 'Aceptar')]").click()
        sleep(1)
    except (NoSuchElementException, ElementNotInteractableException):
        pass

def error_400():
    return "Request Too Long" in driver.page_source or "Bad Request" in driver.title

for pagina in range(PAGINA_INICIO, MAX_PAGINAS + 1):
    url_pagina = f'{URL_BUSQUEDA}?p={pagina}'
    print(f"Página {pagina}: {url_pagina}")

    # Limpiar cookies cada 10 páginas para evitar el 400 por headers largos
    if pagina % 10 == 0:
        driver.delete_all_cookies()

    cargada = False
    for intento in range(3):
        try:
            driver.get(url_pagina)
            sleep(3)
            if error_400():
                print(f"  Error 400, limpiando cookies (intento {intento+1}/3)")
                driver.delete_all_cookies()
                sleep(5)
                continue
            cargada = True
            break
        except Exception as e:
            print(f"  {type(e).__name__}, reintento {intento+1}/3")
            sleep(10 * (intento + 1))
    if not cargada:
        print("  No se pudo cargar. Se detiene; retoma con PAGINA_INICIO =", pagina)
        break

    if pagina == PAGINA_INICIO or pagina % 10 == 0:
        cerrar_banner()

    soup = bs(driver.page_source, 'html.parser')
    tarjetas = soup.select('article.box_offer')

    if not tarjetas:
        print("  Sin ofertas, fin de la paginación.")
        break

    nuevos = 0
    for tarjeta in tarjetas:
        a = tarjeta.select_one('h2 a.js-o-link') or tarjeta.select_one('a.js-o-link')
        if a and a.get('href'):
            link = urljoin(BASE_URL, a['href'])
            if link not in vistos:
                vistos.add(link)
                urls_empleos.append(link)
                nuevos += 1

    print(f"  {len(tarjetas)} tarjetas, {nuevos} nuevas (total {len(urls_empleos)})")

    if nuevos == 0:
        print("  Solo links repetidos, fin de la paginación.")
        break

    sleep(random.uniform(1.5, 2.5))

if LIMITE_EXTRACCION is not None:
    urls_empleos = urls_empleos[:LIMITE_EXTRACCION]

print(f"\nSe extraerán {len(urls_empleos)} ofertas.")

Página 1: https://pe.computrabajo.com/empleos-de-ingenieria-y-tecnico-en-lima?p=1
  20 tarjetas, 20 nuevas (total 20)
Página 2: https://pe.computrabajo.com/empleos-de-ingenieria-y-tecnico-en-lima?p=2
  20 tarjetas, 20 nuevas (total 40)
Página 3: https://pe.computrabajo.com/empleos-de-ingenieria-y-tecnico-en-lima?p=3
  20 tarjetas, 20 nuevas (total 60)
Página 4: https://pe.computrabajo.com/empleos-de-ingenieria-y-tecnico-en-lima?p=4
  20 tarjetas, 20 nuevas (total 80)
Página 5: https://pe.computrabajo.com/empleos-de-ingenieria-y-tecnico-en-lima?p=5
  20 tarjetas, 20 nuevas (total 100)
Página 6: https://pe.computrabajo.com/empleos-de-ingenieria-y-tecnico-en-lima?p=6
  20 tarjetas, 20 nuevas (total 120)
Página 7: https://pe.computrabajo.com/empleos-de-ingenieria-y-tecnico-en-lima?p=7
  20 tarjetas, 20 nuevas (total 140)
Página 8: https://pe.computrabajo.com/empleos-de-ingenieria-y-tecnico-en-lima?p=8
  20 tarjetas, 20 nuevas (total 160)
Página 9: https://pe.computrabajo.com/empleos-de-ing

In [6]:
resultados = []

try:
    for i, url_empleo in enumerate(urls_empleos):
        print(f"[{i+1}/{len(urls_empleos)}] Extrayendo datos de: {url_empleo}")
        if i % 20 == 0:
            driver.delete_all_cookies()
        driver.get(url_empleo)
        if "Request Too Long" in driver.page_source:
            driver.delete_all_cookies()
            sleep(3)
            driver.get(url_empleo)

        try:
            WebDriverWait(driver, 8).until(EC.presence_of_element_located((By.TAG_NAME, 'h1')))
        except TimeoutException:
            print("⚠️ La página tardó en cargar. Tienes 20 segundos por si aparece alguna verificación...")
            try:
                WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.TAG_NAME, 'h1')))
                print("✅ Contenido cargado. Continuando extracción...")
            except TimeoutException:
                print(f"❌ No cargó a tiempo. Guardando error_empleo_{i+1}.png")
                driver.save_screenshot(f"error_empleo_{i+1}.png")
                continue

        html = driver.page_source
        soup_emp = bs(html, 'html.parser')

        h1 = soup_emp.find('h1')
        titulo = h1.get_text(strip=True) if h1 else "No especificado"

        bloques = dividir_bloques(texto_visible(html), titulo)

        descripcion = bloques["descripcion"] or "No especificado"

        requisitos = bloques["requisitos"]
        if not requisitos:
            m = re.search(r'(requisitos?[^\n]*\n(?:.+\n?)*)', bloques["descripcion"], re.IGNORECASE)
            requisitos = m.group(1).strip() if m else None
        requisitos = requisitos or "No especificado"

        salario = buscar_con_prioridad(PATRON_SALARIO, bloques["resumen"], bloques["descripcion"]) or "No especificado"
        modalidad = buscar_con_prioridad(PATRON_MODALIDAD, bloques["resumen"], bloques["descripcion"]) or "No especificado"

        resultados.append({
            "titulo": titulo,
            "descripcion": descripcion,
            "requisitos": requisitos,
            "salario": salario,
            "ubicacion": "Lima, Perú",
            "modalidad": modalidad,
            "url": url_empleo,
        })
        print(f"✓ Datos extraídos con éxito: {titulo}")

        sleep(3)
except KeyboardInterrupt:
    print("Interrumpido: se conservan los resultados ya extraídos.")
finally:
    driver.quit()

print(f"Total extraído: {len(resultados)}")

[1/740] Extrayendo datos de: https://pe.computrabajo.com/ofertas-de-trabajo/oferta-de-trabajo-de-coordinador-de-control-de-calidad-trasformadores-en-lima-5BF5124F47936B3061373E686DCF3405#lc=ListOffers-Score-0
✓ Datos extraídos con éxito: Coordinador de control de calidad - Trasformadores
[2/740] Extrayendo datos de: https://pe.computrabajo.com/ofertas-de-trabajo/oferta-de-trabajo-de-disenador-mecanico-en-lima-3165BAF0C30A1D9C61373E686DCF3405#lc=ListOffers-Score-1
✓ Datos extraídos con éxito: Diseñador mecánico
[3/740] Extrayendo datos de: https://pe.computrabajo.com/ofertas-de-trabajo/oferta-de-trabajo-de-asistente-de-modelos-de-datos-riesgo-estructural-en-magdalena-del-mar-2D37E91491A0C29661373E686DCF3405#lc=ListOffers-Score-2
✓ Datos extraídos con éxito: Asistente de Modelos de Datos - Riesgo Estructural
[4/740] Extrayendo datos de: https://pe.computrabajo.com/ofertas-de-trabajo/oferta-de-trabajo-de-practicante-de-ingenieria-revit-en-santiago-de-surco-931CE3A1B886457D61373E686DCF3405

[29/740] Extrayendo datos de: https://pe.computrabajo.com/ofertas-de-trabajo/oferta-de-trabajo-de-asistente-de-operaciones-sector-hidrocarburos-la-victoria-en-la-victoria-F9B8633C3F981B2861373E686DCF3405#lc=ListOffers-Score-8
✓ Datos extraídos con éxito: Asistente de Operaciones (sector Hidrocarburos) - La Victoria
[30/740] Extrayendo datos de: https://pe.computrabajo.com/ofertas-de-trabajo/oferta-de-trabajo-de-analista-de-operaciones-en-la-victoria-0C630FFA2B2E864B61373E686DCF3405#lc=ListOffers-Score-9
✓ Datos extraídos con éxito: Analista de operaciones
[31/740] Extrayendo datos de: https://pe.computrabajo.com/ofertas-de-trabajo/oferta-de-trabajo-de-software-engineer-genexus-en-san-isidro-607ABE27467D1AEA61373E686DCF3405#lc=ListOffers-Score-10
✓ Datos extraídos con éxito: Software Engineer/ Genexus
[32/740] Extrayendo datos de: https://pe.computrabajo.com/ofertas-de-trabajo/oferta-de-trabajo-de-administradora-de-contrato-facility-management-exp-oficinas-corporativas-disponibilidad-in

✓ Datos extraídos con éxito: Analista de costos Bachiller - Civil, Industrial, Contabilidad o Economía
[58/740] Extrayendo datos de: https://pe.computrabajo.com/ofertas-de-trabajo/oferta-de-trabajo-de-coordinadora-de-mantenimiento-servicios-industriales-y-generales-en-lima-1EDECF2B882A68E661373E686DCF3405#lc=ListOffers-Score-17
✓ Datos extraídos con éxito: Coordinador(a) de Mantenimiento - Servicios Industriales y Generales
[59/740] Extrayendo datos de: https://pe.computrabajo.com/ofertas-de-trabajo/oferta-de-trabajo-de-supervisor-civil-proyecto-linea-2-en-san-juan-de-miraflores-6980136AE84B7C9361373E686DCF3405#lc=ListOffers-Score-18
✓ Datos extraídos con éxito: Supervisor Civil / Proyecto Linea 2
[60/740] Extrayendo datos de: https://pe.computrabajo.com/ofertas-de-trabajo/oferta-de-trabajo-de-ingeniero-coordinador7-anos-de-experiencia-en-lima-808611844000CC1661373E686DCF3405#lc=ListOffers-Score-19
✓ Datos extraídos con éxito: Ingeniero Coordinador/7 años de experiencia
[61/740] Extray

[85/740] Extrayendo datos de: https://pe.computrabajo.com/ofertas-de-trabajo/oferta-de-trabajo-de-ingeniero-instructor-en-riesgos-criticos-ingenieria-en-san-isidro-8C1AE5359F5854F361373E686DCF3405#lc=ListOffers-Score-4
✓ Datos extraídos con éxito: Ingeniero Instructor en riesgos críticos - Ingeniería
[86/740] Extrayendo datos de: https://pe.computrabajo.com/ofertas-de-trabajo/oferta-de-trabajo-de-ingenieroa-geotecnico-por-6-meses-sector-mineria-surco-lima-ref-112071-en-santiago-de-surco-B377EFDFD9F4EA1D61373E686DCF3405#lc=ListOffers-Score-5
✓ Datos extraídos con éxito: Ingeniero(a) Geotécnico por 6 meses / Sector minería / Surco, Lima  (Ref: 112071)
[87/740] Extrayendo datos de: https://pe.computrabajo.com/ofertas-de-trabajo/oferta-de-trabajo-de-tecnico-de-servicio-servicio-postventa-en-surquillo-6F6D7A26CCCB9B9E61373E686DCF3405#lc=ListOffers-Score-6
✓ Datos extraídos con éxito: Técnico de Servicio - Servicio PostVenta
[88/740] Extrayendo datos de: https://pe.computrabajo.com/ofertas-d

✓ Datos extraídos con éxito: Analista de costos y presupuestos
[112/740] Extrayendo datos de: https://pe.computrabajo.com/ofertas-de-trabajo/oferta-de-trabajo-de-ingeniero-de-seguridad-mina-subterranea-en-lima-C685EA128316EAFF61373E686DCF3405#lc=ListOffers-Score-11
✓ Datos extraídos con éxito: Ingeniero de Seguridad Mina Subterránea
[113/740] Extrayendo datos de: https://pe.computrabajo.com/ofertas-de-trabajo/oferta-de-trabajo-de-planificador-de-mantenimiento-jr-san-juan-de-lurigancho-en-lima-2944BC7C7DEB6DF961373E686DCF3405#lc=ListOffers-Score-12
✓ Datos extraídos con éxito: Planificador de Mantenimiento Jr - San Juan de Lurigancho
[114/740] Extrayendo datos de: https://pe.computrabajo.com/ofertas-de-trabajo/oferta-de-trabajo-de-asistente-de-mejora-continua-metodologias-5s-y-tpm-sjl-rubro-industrial-en-lima-61B9F01BF392068461373E686DCF3405#lc=ListOffers-Score-13
✓ Datos extraídos con éxito: Asistente de Mejora Continua / Metodologías 5s y TPM - SJL Rubro industrial
[115/740] Extrayend

✓ Datos extraídos con éxito: Coordinador de Imágenes Medicas - San Juan de Miraflores
[140/740] Extrayendo datos de: https://pe.computrabajo.com/ofertas-de-trabajo/oferta-de-trabajo-de-ingeniero-electromecanico-en-santiago-de-surco-4445AAF35F9B1C6161373E686DCF3405#lc=ListOffers-Score-19
✓ Datos extraídos con éxito: Ingeniero electromecánico
[141/740] Extrayendo datos de: https://pe.computrabajo.com/ofertas-de-trabajo/oferta-de-trabajo-de-asistente-de-produccion-costura-sede-ate-planilla-completa-utilidades-en-ate-64000D7D2BE55B6661373E686DCF3405#lc=ListOffers-Score-0
✓ Datos extraídos con éxito: Asistente de Producción / Costura / sede Ate - Planilla Completa/ Utilidades
[142/740] Extrayendo datos de: https://pe.computrabajo.com/ofertas-de-trabajo/oferta-de-trabajo-de-desarrollador-de-prototipos-tecnologicos-arduino-iot-e-inteligencia-artificial-en-lince-4B96651D991AE03761373E686DCF3405#lc=ListOffers-Score-1
✓ Datos extraídos con éxito: Desarrollador de Prototipos Tecnológicos – Arduin

... (17 outputs left out)

[644/740] Extrayendo datos de: https://pe.computrabajo.com/ofertas-de-trabajo/oferta-de-trabajo-de-ingeniero-estructural-en-santiago-de-surco-B48B90AE9CBB652261373E686DCF3405#lc=ListOffers-Score-3
✓ Datos extraídos con éxito: Ingeniero Estructural
[645/740] Extrayendo datos de: https://pe.computrabajo.com/ofertas-de-trabajo/oferta-de-trabajo-de-ingeniero-ssoma-para-servicios-puntuales-en-lima-515F5AC575E1F07861373E686DCF3405#lc=ListOffers-Score-4
✓ Datos extraídos con éxito: Ingeniero ssoma para servicios puntuales
[646/740] Extrayendo datos de: https://pe.computrabajo.com/ofertas-de-trabajo/oferta-de-trabajo-de-oficial-armadora-calderas-y-caldereria-en-lurin-111641B5A0A8AB0361373E686DCF3405#lc=ListOffers-Score-5
✓ Datos extraídos con éxito: oficial armador/a - Calderas y Caldereria
[647/740] Extrayendo datos de: https://pe.computrabajo.com/ofertas-de-trabajo/oferta-de-trabajo-de-tornero-con-experiencia-en-carabayllo-630D80461485C31861373E686DCF3405#lc=ListOffers-Score-6
✓ Datos extraí

[674/740] Extrayendo datos de: https://pe.computrabajo.com/ofertas-de-trabajo/oferta-de-trabajo-de-ingeniero-preventa-en-sistemas-de-seguridad-electronica-y-automatizacion-en-miraflores-AFC92C291F762F0061373E686DCF3405#lc=ListOffers-Score-13
✓ Datos extraídos con éxito: Ingeniero Preventa en sistemas de Seguridad Electrónica y Automatización
[675/740] Extrayendo datos de: https://pe.computrabajo.com/ofertas-de-trabajo/oferta-de-trabajo-de-arquitecto-titulado-en-miraflores-4BD07614D685333E61373E686DCF3405#lc=ListOffers-Score-14
✓ Datos extraídos con éxito: Arquitecto - Titulado
[676/740] Extrayendo datos de: https://pe.computrabajo.com/ofertas-de-trabajo/oferta-de-trabajo-de-ingenierao-quimico-coordinador-con-experiencia-en-los-olivos-0189FAFCD8F6AA4361373E686DCF3405#lc=ListOffers-Score-15
✓ Datos extraídos con éxito: Ingeniera(o) químico - coordinador con experiencia
[677/740] Extrayendo datos de: https://pe.computrabajo.com/ofertas-de-trabajo/oferta-de-trabajo-de-modeladores-dibujante

✓ Datos extraídos con éxito: dibujante mecánico
[702/740] Extrayendo datos de: https://pe.computrabajo.com/ofertas-de-trabajo/oferta-de-trabajo-de-asistente-ssoma-en-ate-8DC2F0C577A7974F61373E686DCF3405#lc=ListOffers-Score-1
✓ Datos extraídos con éxito: asistente ssoma
[703/740] Extrayendo datos de: https://pe.computrabajo.com/ofertas-de-trabajo/oferta-de-trabajo-de-tecnico-de-mantenimiento-en-los-olivos-50D7491DB25ED1C761373E686DCF3405#lc=ListOffers-Score-2
✓ Datos extraídos con éxito: Tecnico de mantenimiento
[704/740] Extrayendo datos de: https://pe.computrabajo.com/ofertas-de-trabajo/oferta-de-trabajo-de-asistente-de-ingenieria-con-conocimientos-en-proyectos-hvac-en-comas-597F1AF648FB4C5661373E686DCF3405#lc=ListOffers-Score-3
✓ Datos extraídos con éxito: Asistente de Ingeniería con conocimientos en proyectos HVAC
[705/740] Extrayendo datos de: https://pe.computrabajo.com/ofertas-de-trabajo/oferta-de-trabajo-de-ingeniero-electrico-junior-con-experiencia-en-instalaciones-y-proyecto-e

✓ Datos extraídos con éxito: Ingeniera Biomédica - Con experiencia
[731/740] Extrayendo datos de: https://pe.computrabajo.com/ofertas-de-trabajo/oferta-de-trabajo-de-ingeniero-electricista-en-lima-6E49F4A687395D7761373E686DCF3405#lc=ListOffers-Score-10
✓ Datos extraídos con éxito: ingeniero electricista
[732/740] Extrayendo datos de: https://pe.computrabajo.com/ofertas-de-trabajo/oferta-de-trabajo-de-asistente-de-ssoma-gestion-documentaria-y-reportes-en-puente-piedra-264B30679F79D46861373E686DCF3405#lc=ListOffers-Score-11
✓ Datos extraídos con éxito: asistente de ssoma - Gestión Documentaria y Reportes
[733/740] Extrayendo datos de: https://pe.computrabajo.com/ofertas-de-trabajo/oferta-de-trabajo-de-lider-de-proyectos-sistema-integrado-de-gestion-en-miraflores-99FF8070886E714161373E686DCF3405#lc=ListOffers-Score-12
✓ Datos extraídos con éxito: Líder de Proyectos - Sistema Integrado de Gestión
[734/740] Extrayendo datos de: https://pe.computrabajo.com/ofertas-de-trabajo/oferta-de-trabaj

In [7]:
df_final = pd.DataFrame(resultados)
df_final.head()

,titulo,descripcion,requisitos,salario,ubicacion,modalidad,url
0,Coordinador de control de calidad - Trasformad...,"S/. 1.800,00 (Mensual)\nContrato por Obra Dete...",Tecnico o Bachiller de Ingenieria Electrica o ...,"S/. 1.800,00","Lima, Perú",No especificado,https://pe.computrabajo.com/ofertas-de-trabajo...
1,Diseñador mecánico,A convenir\nContrato por Inicio o Incremento d...,Bachiller en Ingeniería Mecánica o carreras af...,No especificado,"Lima, Perú",No especificado,https://pe.computrabajo.com/ofertas-de-trabajo...
2,Asistente de Modelos de Datos - Riesgo Estruct...,A convenir\nContrato por Necesidades del Merca...,patrimoniales.\nEvaluar políticas de productos...,No especificado,"Lima, Perú",No especificado,https://pe.computrabajo.com/ofertas-de-trabajo...
3,Practicante de ingeniería - Revit,"S/. 1.250,00 (Mensual)\nConvenio de Prácticas\...",Educación mínima: Educación Secundaria\naños d...,"S/. 1.250,00","Lima, Perú",No especificado,https://pe.computrabajo.com/ofertas-de-trabajo...
4,Coordinador SIG y SSoma,"S/. 2.500,00 (Mensual)\nContrato por Obra Dete...",":\n•\tTitulado, bachiller o egresado(a) de Ing...","S/. 2.500,00","Lima, Perú",No especificado,https://pe.computrabajo.com/ofertas-de-trabajo...


In [8]:
df_final.to_excel("empleos_computrabajo.xlsx", index=False)
print("📊 Datos exportados a: empleos_computrabajo.xlsx")

📊 Datos exportados a: empleos_computrabajo.xlsx
